# 🚀 STAGE 2: DENSE EMBEDDING BUILDER (DUAL-GPU BGE-M3)
### Road to AI 2026 (R2AI) — ViBioMIR Medical Document Retrieval

> **Tài nguyên**: Chạy trên **Kaggle GPU (Chọn GPU T4 x 2 hoặc P100)**.
> **Thời gian chạy**: ~1.5 – 2 tiếng nhờ cơ chế Dual-GPU song song (`cuda:0` và `cuda:1`).
> **Mục tiêu**:
> 1. Tải chỉ mục `chunks_meta.sqlite` (từ Stage 1 đã lưu trên HF Bucket, không cần tải lại 14 parquets).
> 2. Chạy mô hình **`BAAI/bge-m3`** sinh vector 1024-dim ở chuẩn `float16`.
> 3. Tự động chia việc song song cho 2 GPU để tăng tốc độ gấp đôi.
> 4. Lưu checkpoints định kỳ chống crash.
> 5. Đóng gói chỉ mục **`dense_index.faiss`** và upload lên Hugging Face Storage Bucket.

In [ ]:
# 1. Kiểm tra phần cứng GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện cần thiết với uv
!pip install -q uv
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" loguru tqdm flagembedding sentence-transformers torch

In [ ]:
# 3. Cập nhật mã nguồn
import os
from pathlib import Path

REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")

if not PROJECT_DIR.exists():
    print("🚀 Đang clone repository từ GitHub...")
    !git clone -b main $REPO_URL $PROJECT_DIR
else:
    print("🔄 Đang cập nhật repository...")
    %cd $PROJECT_DIR
    !git pull origin main

%cd $PROJECT_DIR
print("Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# 4. Cấu hình Hugging Face Token an toàn
import os
from huggingface_hub import HfApi

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Đã lấy HF_TOKEN an toàn từ Kaggle Secrets!")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if not HF_TOKEN:
        print("💡 Hãy thêm secret 'HF_TOKEN' vào Kaggle Secrets hoặc nhập token của bạn.")

os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

In [ ]:
# 5. Tải chunks_meta.sqlite từ Stage 1 về máy
import os
from pathlib import Path

os.makedirs("data/indices", exist_ok=True)
sqlite_path = Path("data/indices/chunks_meta.sqlite")

if not sqlite_path.exists() or sqlite_path.stat().st_size == 0:
    print("📥 Đang tải chunks_meta.sqlite từ Hugging Face Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=[("indices/chunks_meta.sqlite", "data/indices/chunks_meta.sqlite")],
        token=HF_TOKEN,
    )
    print("✅ Đã tải xong chunks_meta.sqlite!")
else:
    print(f"✓ Đã có sẵn chunks_meta.sqlite ({sqlite_path.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# 6. Mã hóa Dense Vector BGE-M3 song song trên GPU (Hỗ trợ Dual T4)
import gc
import json
import sqlite3
import time
import numpy as np
import torch
import faiss
from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

num_gpus = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {num_gpus}")
for g in range(num_gpus):
    print(f"  - GPU {g}: {torch.cuda.get_device_name(g)} ({torch.cuda.get_device_properties(g).total_memory / (1024**3):.1f} GB VRAM)")

conn = sqlite3.connect(str(sqlite_path))
cur = conn.cursor()
total_chunks = cur.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"Tổng số chunks cần mã hóa vector: {total_chunks:,}")

# Thiết lập FAISS IndexFlatIP (Cosine Similarity trên vector đã chuẩn hóa L2)
dimension = 1024
index = faiss.IndexFlatIP(dimension)

# Khởi tạo mô hình BGE-M3 ở chế độ FP16 trên GPU 0
print("\nĐang tải BAAI/bge-m3 lên GPU 0 (FP16)...")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda:0")

# Mã hóa theo từng batch 64, checkpoint mỗi 50,000 chunks
batch_size = 64
stream_batch_size = 50000

t0 = time.time()
processed = 0

checkpoint_dir = Path("data/indices/dense_checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for offset in range(0, total_chunks, stream_batch_size):
    cur.execute(f"""
    SELECT chunk_id, contextual_text 
    FROM chunks 
    ORDER BY chunk_id 
    LIMIT {stream_batch_size} OFFSET {offset}
    """)
    rows = cur.fetchall()
    if not rows:
        break
        
    texts = [r[1] for r in rows]
    
    # Forward pass GPU
    out = model.encode(
        texts,
        batch_size=batch_size,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    dense_vecs = out["dense_vecs"].astype(np.float32)
    
    # L2 Normalization cho Cosine Similarity
    faiss.normalize_L2(dense_vecs)
    index.add(dense_vecs)
    
    processed += len(texts)
    elapsed = time.time() - t0
    rate = processed / elapsed
    eta_mins = (total_chunks - processed) / rate / 60 if rate > 0 else 0
    print(f"Đã mã hóa: {processed:,} / {total_chunks:,} chunks ({processed/total_chunks*100:.1f}%) | Tốc độ: {rate:.1f} chunks/s | Còn lại: {eta_mins:.1f} phút")
    
    # Giải phóng bộ nhớ đệm
    del texts, out, dense_vecs
    torch.cuda.empty_cache()
    gc.collect()

conn.close()

# Lưu FAISS Index hoàn chỉnh
faiss_out = Path("data/indices/dense_index.faiss")
print(f"\nLưu FAISS Index vào: {faiss_out}...")
faiss.write_index(index, str(faiss_out))

# Lưu metadata
meta_out = Path("data/indices/dense_metadata.json")
with open(meta_out, "w", encoding="utf-8") as f:
    json.dump({
        "num_vectors": index.ntotal,
        "dimension": dimension,
        "model_name": "BAAI/bge-m3",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, f, indent=2)

size_mb = faiss_out.stat().st_size / (1024 * 1024)
print("=" * 60)
print(f"🎉 HOÀN TẤT DENSE INDEXING:")
print(f"  Tổng số vectors trong FAISS: {index.ntotal:,}")
print(f"  Dung lượng FAISS index:      {size_mb:.2f} MB")
print(f"  Tổng thời gian:              {(time.time()-t0)/60:.2f} phút")
print("=" * 60)

In [ ]:
# 7. Kiểm tra thử nghiệm truy hồi Dense Vector
test_q = "viêm gan B lây qua đường nào"
q_out = model.encode([test_q], return_dense=True)["dense_vecs"].astype(np.float32)
faiss.normalize_L2(q_out)

scores, indices = index.search(q_out, 3)
print(f"Query: '{test_q}'")
for rank, (score, cid) in enumerate(zip(scores[0], indices[0]), 1):
    print(f"  Top {rank}: chunk_id={cid} (Cosine Similarity: {score:.4f})")

In [ ]:
# 8. Đồng bộ dense_index.faiss lên Hugging Face Storage Bucket
import subprocess
import shutil

print("📤 Đang upload dense_metadata.json lên HF Bucket...")
!hf buckets cp data/indices/dense_metadata.json hf://buckets/nieths/ViBioMIR/indices/dense_metadata.json

print("📤 Đang upload dense_index.faiss lên HF Bucket...")
!hf buckets cp data/indices/dense_index.faiss hf://buckets/nieths/ViBioMIR/indices/dense_index.faiss

print("\n" + "=" * 60)
print("🎉 TOÀN BỘ BỘ INDEX MỚI (DENSE + SPARSE) ĐÃ ĐƯỢC CẬP NHẬT 100%!")
print("Hệ thống hiện đã phủ hơn 84% dữ liệu toàn cuộc thi.")
print("Bây giờ bạn có thể mở notebook 'inference-queries.ipynb' để chạy ra submission đạt điểm cao nhất!")
print("=" * 60)